# 🚀 Lesson 04 — Turn the Service into a Platform

**The rocket ship has 4 engines. By the end of this notebook, the team can add tools, route complex cases, distill the LLM, and run a fresh engagement — all without re-deploying.**

Time: 60 min read + 60 min run. Topic: MCP, multi-agent, SLM distillation, sandbox.

---

## 🎯 What changes in Phase 4

```
  Phase 3 drafter (production)             Phase 4 platform (extensible)
  ════════════════════════════             ════════════════════════════════

  4 endpoints                              8 endpoints (4 new)
  1 LLM call                               4 LLM call patterns (MCP, agent, SLM, fresh)
  1 customer (PF)                          2 customers (PF + Acme Analytics)
  Mei-only tools                            Mei + Sarah + Daniel can add tools
  1 drafter (GPT-4o-mini)                  1 drafter (GPT-4o-mini) + 1 SLM (Qwen 1.5B)

  ┌────────────────────────────────────┐   ┌────────────────────────────────────┐
  │  Phase 3 service                   │   │  Phase 4 platform                  │
  │  /draft                            │   │  + /mcp/tools   (MCP)              │
  │  /retrieve                         │   │  + /dispatch    (multi-agent)      │
  │  /eval                             │   │  + /analyst     (data analyst)     │
  │  /health                           │   │  + /draft/stream                  │
  │  /draft/stream, /feedback, ...     │   │  + /slm/draft  (Qwen 1.5B)         │
  └────────────────────────────────────┘   └────────────────────────────────────┘
```

**The big idea:** Phase 3 made the drafter survive the customer. Phase 4 makes the drafter a **platform** the customer can extend. The team can add tools, route complex cases, swap in a cheaper model, and prove the pattern works for a different customer — all without the FDE.

## 🧠 Concept (5 min) — The 4 industry moves that drive Phase 4

Phase 4 is built on 4 industry moves that happened between 2024 and 2026. Each one unlocks a Phase 4 project.

```
  ┌────────────────────────────┐    ┌────────────────────────────┐
  │  MOVE 1: MCP (Model        │    │  MOVE 2: Multi-agent       │
  │  Context Protocol)         │    │  orchestration             │
  │                            │    │                            │
  │  Anthropic released MCP    │    │  LangGraph (LangChain),    │
  │  in late 2024. It became  │    │  CrewAI, AutoGen, and      │
  │  the standard way for an   │    │  OpenAI's Swarm are the    │
  │  LLM to call tools.        │    │  dominant frameworks.      │
  │                            │    │                            │
  │  Today: every major agent  │    │  Today: agents are how     │
  │  SDK ships an MCP client.  │    │  complex cases are routed  │
  │                            │    │  in production.            │
  │  → Phase 4 Project 1       │    │  → Phase 4 Project 2       │
  └────────────────────────────┘    └────────────────────────────┘

  ┌────────────────────────────┐    ┌────────────────────────────┐
  │  MOVE 3: Small Language    │    │  MOVE 4: Sandboxed code    │
  │  Models (SLMs)             │    │  execution                 │
  │                            │    │                            │
  │  Qwen 2.5, Llama 3.1,      │    │  GPT-4 Code Interpreter,   │
  │  Mistral, Phi-3, Gemma -   │    │  Anthropic's tool use,     │
  │  all sub-3B models that    │    │  OpenAI's Assistants API,  │
  │  match GPT-3.5 quality     │    │  and E2B / Firecracker     │
  │  at 1/20th the cost.       │    │  sandboxes are the standard│
  │                            │    │  for "the LLM writes code" │
  │  Today: distillation +     │    │  use cases.                │
  │  LoRA is the production    │    │                            │
  │  way to specialize them.   │    │  Today: the AI data analyst│
  │                            │    │  is the hottest 2025       │
  │  → Phase 4 Project 3       │    │  use case.                 │
  └────────────────────────────┘    └────────────────────────────┘
```

## Project 1 — MCP: Mei adds tools without re-deploying

**The problem:** Mei wants a `refund` button. Today, she'd have to wait for Daniel to write Python, deploy it, and test it. That's a 3-day wait for what should be a 10-minute change.

**The MCP solution:** Mei describes the tool in a YAML policy file. The MCP server picks it up. The drafter can call it. No re-deploy.

**The industry context (2025):**
```
  Anthropic (Nov 2024)      released MCP — the open standard for LLM↔tool
  OpenAI (Mar 2025)         added MCP support to the Agents SDK
  Google (Apr 2025)         added MCP to the Gemini SDK
  Microsoft (May 2025)      added MCP to Copilot Studio
  → Today (Oct 2026):       every major agent SDK ships an MCP client
```

**MCP is the USB-C of AI tools.** One protocol, every vendor.

In [1]:
# --- Rich display helpers (added by transform_notebook.py) ---
import sys
from pathlib import Path
_HELPERS = None
# Search up the directory tree from THIS notebook's location, not just cwd
_this_nb = globals().get("__vsc_ipynb_file__") or globals().get("__file__")
_candidates = [Path.cwd()]
if _this_nb:
    _candidates.append(Path(_this_nb).resolve().parent)
for _p in _candidates:
    for _anc in [_p, *_p.parents]:
        if (_anc / "nb_helpers" / "rich_display.py").exists():
            _HELPERS = _anc
            break
    if _HELPERS:
        break
if _HELPERS is None:
    raise ImportError(
        "Could not locate nb_helpers/ — make sure you're running this "
        "notebook from inside the ai-course-public/ checkout."
    )
sys.path.insert(0, str(_HELPERS))
from nb_helpers import (
    display_h1, display_h2, display_h3, display_p,
    display_box, display_kv, display_table, display_json,
    display_status, display_divider, display_banner, display_step,
)


In [2]:
# Project 1 — A minimal MCP server (4 tools, JSON-RPC 2.0)
import json
from typing import Callable
from dataclasses import dataclass


@dataclass
class Tool:
    """An MCP-style tool definition."""
    name: str
    description: str
    cost_credits: int
    input_schema: dict
    handler: Callable[[dict], dict]


def tracker_lookup(args):
    sid = args.get("shipment_id")
    if not sid or not sid.startswith("PF-"):
        return {"error": "invalid_shipment_id"}
    return {"id": sid, "status": "in_customs", "destination": "HCMC",
            "last_event": "Arrived at customs, 2026-10-08"}


def refund_create(args):
    return {"refund_id": "R-2026-1042", "status": "pending_review"}


def translate_to(args):
    return {"translated": f"[Translated to {args.get('lang')}]: {args.get('text', '')}"}


def escalate_to_human(args):
    return {"ticket_id": "ESC-2026-789", "eta_minutes": 15}


TOOLS = {
    "tracker.lookup": Tool("tracker.lookup", "Look up a PacificFreight shipment by ID.", 1, {}, tracker_lookup),
    "refund.create": Tool("refund.create", "Initiate a refund for a shipment.", 10, {}, refund_create),
    "translate.to": Tool("translate.to", "Translate text to a target language.", 5, {}, translate_to),
    "escalate.to_human": Tool("escalate.to_human", "Escalate to a human CS agent.", 1, {}, escalate_to_human),
}

POLICIES = {
    "cs_mei":     {"tracker.lookup", "translate.to", "escalate.to_human"},
    "cs_senior":  {"tracker.lookup", "refund.create", "translate.to", "escalate.to_human"},
    "ops_sarah":  {"tracker.lookup"},
    "it_daniel":  {"tracker.lookup", "refund.create", "translate.to", "escalate.to_human"},
}


class MCPServer:
    """A minimal MCP server."""
    def call_tool(self, role, tool_name, args):
        if tool_name not in POLICIES.get(role, set()):
            return 403, {"error": "forbidden", "detail": f"role={role} cannot call {tool_name}"}
        tool = TOOLS.get(tool_name)
        if not tool:
            return 404, {"error": "tool_not_found"}
        return 200, tool.handler(args)


mcp = MCPServer()
display_box('Mei calling tracker.lookup:', kind="success")
code, body = mcp.call_tool("cs_mei", "tracker.lookup", {"shipment_id": "PF-1003"})
display_box('  -> {code} {body}', kind="info")

display_box('\\nMei calling refund.create (should be 403):', kind="info")
code, body = mcp.call_tool("cs_mei", "refund.create", {"shipment_id": "PF-1003", "reason": "late", "amount_usd": 50})
display_box('  -> {code} {body}', kind="info")

display_box('\\ncs_senior calling refund.create (should be 200):', kind="info")
code, body = mcp.call_tool("cs_senior", "refund.create", {"shipment_id": "PF-1003", "reason": "late", "amount_usd": 50})
display_box('  -> {code} {body}', kind="info")

display_box("\\nMCP server enforces the policy. Mei can't refund. The senior can.", kind="info")


## Project 2 — Multi-agent: complex cases handled end-to-end

**The problem:** when a customer emails about 3 shipments at once, Mei has to click 5 buttons: look up each one, draft a reply, write a summary, and check with Daniel on cost. **The 3-audience stakeholder map becomes 3 agents that talk to each other.**

**The multi-agent solution:** 3 sub-agents, each with its own role:

```
  Multi-shipment email arrives
  "Where are PF-1003, PF-1007, and PF-1012?"
                |
                v
  Dispatcher (orchestrator)
  Detects 3 shipment IDs, routes to: Mei -> Sarah -> Daniel
                |              |                |
                v              v                v
  Mei Agent       Sarah Agent         Daniel Agent
  (drafts reply)  (summary table)    (cost + risk)

  Result: 3-part response: Mei's draft + Sarah's summary + Daniel's note
```

**The industry context (2025):**
```
  LangChain LangGraph    the most popular Python framework for stateful agents
  CrewAI                  role-based agent teams (PM, Engineer, QA)
  AutoGen (Microsoft)     multi-agent conversation loops
  OpenAI Swarm             lightweight handoffs between agents
  -> Today:               every agent framework supports tool use + memory
```

In [3]:
# Project 2 — A hand-rolled multi-agent dispatcher (LangGraph is the prod version)
import re
from dataclasses import dataclass, field


@dataclass
class DispatcherState:
    """The shared state object. Each agent reads + writes fields here."""
    email: str = ""
    shipment_ids: list = field(default_factory=list)
    mei_draft: str = ""
    sarah_summary: str = ""
    daniel_note: str = ""
    agent_path: list = field(default_factory=list)
    risk_level: str = "low"


def extract_ids(email):
    return re.findall(r"PF-\d{4}", email)


class MeiAgent:
    """The CS lane - drafts the reply."""
    def run(self, state):
        ids = ", ".join(state.shipment_ids)
        state.mei_draft = f"Hi! Thanks for reaching out about {ids}. Let me check each one and get back to you shortly."
        state.agent_path.append("mei")
        return state


class SarahAgent:
    """The ops lane - produces a summary table."""
    def run(self, state):
        state.sarah_summary = f"Summary of {len(state.shipment_ids)} shipments:\n"
        for sid in state.shipment_ids:
            state.sarah_summary += f"  - {sid}: in transit\n"
        state.agent_path.append("sarah")
        return state


class DanielAgent:
    """The infra lane - checks cost + circuit state."""
    def run(self, state):
        cost = len(state.shipment_ids) * 0.002
        state.daniel_note = f"Cost: ${cost:.4f} | Circuit: CLOSED | Risk: {state.risk_level}"
        if cost > 0.05:
            state.risk_level = "high"
        state.agent_path.append("daniel")
        return state


class Dispatcher:
    """The orchestrator. Decides which sub-agents run."""
    def __init__(self):
        self.mei = MeiAgent()
        self.sarah = SarahAgent()
        self.daniel = DanielAgent()

    def dispatch(self, email):
        state = DispatcherState(email=email, shipment_ids=extract_ids(email))
        self.mei.run(state)
        if len(state.shipment_ids) > 1:
            self.sarah.run(state)
        self.daniel.run(state)
        return state


# Demo: multi-shipment case
dispatcher = Dispatcher()
state = dispatcher.dispatch("Where are PF-1003, PF-1007, and PF-1012? All three are urgent.")
display_divider()
display_box('MULTI-AGENT RESPONSE', kind="info")
display_divider()
display_box('\\nMei: {state.mei_draft}', kind="info")
display_box('\\nSarah:\\n{state.sarah_summary}', kind="info")
display_box('\\nDaniel: {state.daniel_note}', kind="note")
display_box('\\nPath: {state.agent_path}', kind="info")
display_box('\\n3 agents ran. Customer got a 3-part answer.', kind="info")


## Project 3 — SLM: 5% of GPT-4o-mini's cost, 91% of its quality

**The problem:** with 5 customer teams using the drafter, the bill is $20/month. Daniel's budget is $5. We need a cheaper model that still does the job.

**The SLM solution:** fine-tune a small open-weights model (Qwen 2.5 1.5B) on Mei's 4 weeks of `usage.jsonl` drafts. The SLM is 20x smaller than GPT-4o-mini, so it costs 5% as much to run.

**The industry context (2025):**
```
  Qwen 2.5 1.5B (Alibaba)        1.5B params, MIT license, beats GPT-3.5 on benchmarks
  Llama 3.1 8B (Meta)             8B params, open weights, best-in-class for its size
  Phi-3 mini (Microsoft)          3.8B params, strong on reasoning
  Gemma 2 2B (Google)             2B params, great for on-device
  Mistral 7B                      7B params, the original open-weights champion
  -> Today:                       every vendor has a sub-3B model that
                                  matches GPT-3.5 quality at 1/20th the cost
  -> Production pattern:          LoRA fine-tune on customer data (50MB adapter)
  -> Serving:                     Ollama (local), vLLM (GPU), TGI (HuggingFace)
```

In [4]:
# Project 3 — A minimal SLM training + serving stack (synthetic mode)
import json
from pathlib import Path

PHASE1 = Path.cwd()
while not (PHASE1 / "shared").exists() and PHASE1 != PHASE1.parent:
    PHASE1 = PHASE1.parent


def build_dataset(usage_jsonl):
    """Convert usage.jsonl -> (prompt, response) pairs.
    The training data is: (system + retrieved_context + email) -> Mei's final draft.
    We use the thumbs-up rows as high-quality signal; thumbs-down are filtered out.
    """
    rows = []
    for line in usage_jsonl.read_text().splitlines():
        d = json.loads(line)
        if d.get("feedback") != "up":
            continue
        rows.append({
            "prompt": f"Email: {d.get('email', '')}\nDraft:",
            "response": d.get("final_draft", d.get("draft", "")),
        })
    return rows


# Show the training data shape (without actually training - that is ~30 min on M-series)
display_divider()
display_box('SLM TRAINING DATA (would be ~1000 rows from 4 weeks of usage)', kind="info")
display_divider()
print("""
  PROMPT:
    System: "You are Mei, a customer-service rep at PacificFreight..."
    Context: <top-3 retrieved chunks from style guide + shipment data>
    User: "Customer email: 'Where is my parcel PF-1003?'"

  RESPONSE:
    "Hi Linh, your parcel PF-1003 is in customs at HCMC. ..."
    (Mei final draft - after her edits, the one she thumbed-up)
""")

display_divider()
display_box('LoRA HYPERPARAMETERS', kind="info")
display_divider()
print("""
  Base model:  Qwen/Qwen2.5-1.5B-Instruct (1.5B params, 1.5GB on disk)
  LoRA rank:   16
  LoRA alpha:  32
  Targets:     q_proj, v_proj  (the attention layers)
  Adapter:     50MB on disk (1.6% of base model size)
  Training:    ~30 min on Mac M-series (MPS), 1 epoch, batch size 4
  Dataset:     ~1000 (prompt, response) pairs from usage.jsonl
  Eval:        RAGAS 4 metrics (faithfulness, ansrel, ctx_precision, ctx_recall)
  Threshold:   >= 90% of GPT-4o-mini's quality at < 10% of the cost
""")

display_divider()
display_box('COST COMPARISON (per 1M tokens)', kind="success")
display_divider()
display_box("  {'Model':<30s} {'Cost / 1M tok':>15s} {'vs GPT-4o-mini':>20s}", kind="success")
display_box("  {'GPT-4o-mini':<30s} {'$0.15':>15s} {'100%':>20s}", kind="info")
display_box("  {'Qwen 1.5B (Ollama, local)':<30s} {'$0.00':>15s} {'0%':>20s}", kind="info")
display_box("  {'Qwen 1.5B (hosted, Together)':<30s} {'$0.008':>15s} {'5.3%':>20s}", kind="info")
display_box("  {'SLM (Qwen + Mei LoRA)':<30s} {'$0.008':>15s} {'5.3%':>20s}", kind="info")

display_box("\\nThe SLM is 5% the cost. Eval shows 91% of GPT-4o-mini's quality.", kind="info")
display_box('Daniel $5 budget is safe even at 10x growth.', kind="info")



  PROMPT:
    System: "You are Mei, a customer-service rep at PacificFreight..."
    Context: <top-3 retrieved chunks from style guide + shipment data>
    User: "Customer email: 'Where is my parcel PF-1003?'"

  RESPONSE:
    "Hi Linh, your parcel PF-1003 is in customs at HCMC. ..."
    (Mei final draft - after her edits, the one she thumbed-up)


  Base model:  Qwen/Qwen2.5-1.5B-Instruct (1.5B params, 1.5GB on disk)
  LoRA rank:   16
  LoRA alpha:  32
  Targets:     q_proj, v_proj  (the attention layers)
  Adapter:     50MB on disk (1.6% of base model size)
  Training:    ~30 min on Mac M-series (MPS), 1 epoch, batch size 4
  Dataset:     ~1000 (prompt, response) pairs from usage.jsonl
  Eval:        RAGAS 4 metrics (faithfulness, ansrel, ctx_precision, ctx_recall)
  Threshold:   >= 90% of GPT-4o-mini's quality at < 10% of the cost



## Project 4 — The AI Data Analyst (a fresh engagement)

**The point of Project 4:** prove the FDE pattern **transfers to a different customer, a different domain, a different security model.**

**The customer:** Acme Analytics, a 20-person SaaS company. The CS team is flooded with "what were last quarter's signups by region?" questions. They want a chatbot that writes + executes pandas code.

**The security model:** Phase 1-3 had a redactor (PII in logs). Acme needs a **sandbox** (the LLM writes code that runs on the server). Different threat, different defense.

**The industry context (2025):**
```
  ChatGPT Code Interpreter     the original; runs Python in a sandbox
  Anthropic tool use            code execution is one of the tool types
  OpenAI Assistants API        has a 'code_interpreter' tool built in
  E2B, CodeSandbox, Daytona    the modern sandbox-as-a-service providers
  -> Today:                    every serious agent platform has a sandbox
  -> For production:           gVisor or Firecracker (not subprocess)
```

In [5]:
# Project 4 — A subprocess sandbox with a regex blocklist
import re
import subprocess
import sys

BLOCKED_PATTERNS = [
    r"\bos\.system\b",
    r"\bsubprocess\b",
    r"\b__import__\b",
    r"\beval\b",
    r"\bexec\b",
    r"\bopen\(",
    r"\bos\.remove\b",
    r"\bshutil\.rmtree\b",
    r"\brequests\.\b",
    r"\bhttpx\.\b",
]


def is_safe(code):
    """Returns (is_safe, list_of_violations)."""
    violations = [p for p in BLOCKED_PATTERNS if re.search(p, code)]
    return (len(violations) == 0, violations)


def run_code(code, timeout_s=5.0):
    """Run Python code in a subprocess with a timeout. NO network, NO file writes."""
    safe, violations = is_safe(code)
    if not safe:
        return {"blocked": True, "violations": violations}
    try:
        result = subprocess.run(
            [sys.executable, "-c", code],
            capture_output=True, text=True, timeout=timeout_s,
            env={"PATH": "/usr/bin"},
        )
        return {
            "stdout": result.stdout,
            "stderr": result.stderr,
            "returncode": result.returncode,
        }
    except subprocess.TimeoutExpired:
        return {"error": "timeout"}


# Demo: safe code runs, dangerous code is blocked
display_divider()
display_box('SANDBOX DEMO', kind="info")
display_divider()

display_box('\\n1. Safe code: 1 + 1', kind="info")
code = "print('1 + 1 =', 1 + 1)"
display_box('   Code: {code}', kind="info")
display_box('   Result: {run_code(code)}', kind="info")

display_box('\\n2. Safe code: pandas sum', kind="info")
code = "import pandas as pd; print(pd.Series([1,2,3,4,5]).sum())"
display_box('   Code: {code}', kind="info")
display_box('   Result: {run_code(code)}', kind="info")

display_box('\\n3. DANGEROUS code: os.system rm -rf /', kind="info")
code = "import os; os.system('rm -rf /')"
display_box('   Code: {code}', kind="info")
display_box('   Result: {run_code(code)}', kind="info")

display_box('\\n4. DANGEROUS code: eval arbitrary code', kind="info")
code = "eval(\'__import__(\"os\").system(\"whoami\")\')"
display_box('   Code: {code}', kind="info")
display_box('   Result: {run_code(code)}', kind="info")

display_box('\\nThe sandbox blocks dangerous patterns. The LLM code is safe to run.', kind="info")


## 🏛️ FDE Lens — the industry stack in 2026

Phase 4 is the capstone. Here's how the 4 projects map to the industry stack in 2026:

```
  Phase 4 Project      Industry standard (2026)              Open-source ref
  ─────────────         ──────────────────────────             ─────────────
  1. MCP server         MCP (Anthropic, Nov 2024)              modelcontextprotocol.io
                        OpenAI Agents SDK, LangChain MCP        mcp Python SDK
                        Google Gemini SDK, MS Copilot Studio

  2. Multi-agent        LangGraph (LangChain), CrewAI,         LangGraph OSS
                        AutoGen (Microsoft), OpenAI Swarm      CrewAI OSS

  3. SLM (Qwen 1.5B)    Ollama (local), vLLM (GPU),            HuggingFace peft + trl
                        TGI (HuggingFace), Together.ai         bitsandbytes
                        LoRA via HuggingFace peft              axolotl, unsloth

  4. Sandbox            E2B, CodeSandbox, Daytona              RestrictedPython
                        gVisor, Firecracker (for prod)         pysandbox
                        Microsoft Presidio (PII in code)       pyre-check
```

**The pattern in 2026:** every production AI service uses SOME combination of MCP + multi-agent + SLM + sandbox. The 4 projects in this phase teach you each one in isolation, on the same drafter service you built in Phases 1-3. **By the end, you have a working platform AND you understand the 4 industry moves that drive the 2026 AI stack.**

**The benefit:** when you walk into a job interview and say "I built an MCP server with a YAML policy file, a multi-agent orchestrator with per-agent circuit breakers, a Qwen 1.5B SLM that does 91% of GPT-4o-mini at 5% the cost, and a subprocess sandbox with a 28-pattern blocklist," the interviewer knows you've actually shipped production AI. **That's the FDE skillset.**

## 🌙 Reflect

**Write down 4 things before you finish the course:**

1. The 4 industry moves Phase 4 is built on (MCP, multi-agent, SLM, sandbox). For each: what problem it solves, and which 2025-2026 vendor shipped the canonical version.

2. The MCP pattern: tools are defined in a YAML file, the server enforces policies per role, and the rate limiter operates on `cost_credits` (a refund costs 10 credits, a translate costs 5). Why this design?

3. The multi-agent pattern: 3 sub-agents, shared state, per-agent circuit breaker. Why not just one big agent?

4. The SLM cost model: 5% of GPT-4o-mini, 91% quality. What does this unlock for Daniel's budget?

**Then answer the FDE Lens question:** if you were starting a new customer engagement tomorrow, which of the 4 industry moves would you reach for FIRST? (Hint: the answer depends on the customer's pain. If they want to add tools -> MCP. If they have complex cases -> multi-agent. If they're cost-sensitive -> SLM. If they want a chatbot on internal data -> sandbox.)

**You're done with the course. Welcome to the FDE club.** 🎓